# Tools for Forecasting Rat Sightings in Manhattan

Here, we should write down some detailed instructions. Each section follows a similar approach. 

### Instructions for Rat Sightings: 14 Day Forecasts

For this section, the purpose is make a 14 day forecast of rat sightings in Manhattan. The model will make a prediction based on weather data and [data on rat sightings](https://data.cityofnewyork.us/Social-Services/Rat-Sightings/3q43-55fe/about_data) up to the last time the rat sightings dataset was updated. To use, one simply runs all the code blocks in the given section. Here is how it works.

0. One should first set the parameters to be used for the Neural Prophet model. There are some baseline parameters which were obtained by using Optuna. One should change these parameters based on need. Tuning hyperparameters can take quite some time and so we recommend only doing so if the performance of the model is being affected by suboptimal choices of parameters. We will address how to do this below. *To-do: write the Optuna hyperparameter tuning to be done to optimize these parameters once new data is obtained.*

1. Next, it imports needed to make the forecasts. It then download the data on Rat Sightings from [NYC Open Data's Rat Sightings Dataset](https://data.cityofnewyork.us/Social-Services/Rat-Sightings/3q43-55fe/about_data) and also downloads the weather data up to the last day which appears in the Rat Sightings data. For a 14 day forecast, it will assume that the weather on the last day will remain the same for the next 14 days. 

3. Using the weather data and Rat Sightings data, it then trains a Neural Prophet model. **Need to explain how the model works.**

4. Using the trained model, it then forecasts 14 days out. 



### Warnings. 

1. Some fine tuning of the parameters might be needed. We quote from the NeuralProphet documentation: "NeuralProphet is fit with stochastic gradient descent - more precisely, with an AdamW optimizer and a One-Cycle policy. If the parameter learning_rate is not specified, a learning rate range test is conducted to determine the optimal learning rate. The epochs, loss_func and optimizer are other parameters that directly affect the model training process. If not defined, epochsand loss_func are automatically set based on the dataset size. They are set in a manner that controls the total number training steps to be around 1000 to 4000. NeuralProphet offers to set two different values for optimizer, namely AdamW and SDG (stochastic gradient descent). **If it looks like the model is overfitting to the training data (the live loss plot can be useful hereby), you can reduce epochs and learning_rate, and potentially increase the batch_size. If it is underfitting, the number of epochs and learning_rate can be increased and the batch_size potentially decreased.**"

2. The NeuralProphet package appears to no longer be in development. We use the most current version available here.

3. The run time for the modeling process is quite high.

## Rat Sightings: 14 Day Forecasts

### Set Parameters

In [1]:
parameters = dict()
parameters['apparent_temperature_max'] = 54
parameters['apparent_temperature_min'] = 18
parameters['snowfall_sum'] = 1
parameters['n_lags'] = 59 
parameters['epochs'] = 493 
parameters['learning_rate'] = 0.003214767890388168 
parameters['batch_size'] = 220
parameters['ar_reg'] = 0.5847571241076923


parameters['n_forecasts'] = 14 

params = parameters.copy()

The original default settings were:

best_params = dict({'lag_temp_max': 54, 'lag_temp_min': 18, 
                     'lag_snowfall': 1, 'n_lags': 59, 'epochs': 493, 'learning_rate': 0.003214767890388168, 'batch_size': 220, 'ar_reg': 0.5847571241076923})


### Import Packages

In [2]:
import requests 
import os
import shutil
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import datetime

from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import mean_squared_error, mean_absolute_percentage_error
from sklearn.linear_model import LinearRegression

from prophet import Prophet
from pandas.tseries.holiday import USFederalHolidayCalendar
from prophet.diagnostics import cross_validation, performance_metrics
from prophet.plot import plot_cross_validation_metric
from prophet.plot import add_changepoints_to_plot
from prophet.plot import plot_plotly, plot_components_plotly
import itertools

import warnings
from statsmodels.tools.sm_exceptions import ConvergenceWarning
warnings.simplefilter('ignore', ConvergenceWarning)

import optuna
from neuralprophet import NeuralProphet
import logging



/opt/anaconda3/envs/erdos_ds_environment/lib/python3.12/site-packages/lightning_fabric/__init__.py:29: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  __import__("pkg_resources").declare_namespace(__name__)
Importing plotly failed. Interactive plots will not work.
Importing plotly failed. Interactive plots will not work.


In [3]:
## Turn this on if you want to download the most recent data to use. 
## Be careful! The database is updated on a daily basis and so may not be complete.

# # We downloads the rat sightings data to a folder.

# import requests

# url = "https://data.cityofnewyork.us/api/views/3q43-55fe/rows.csv?accessType=DOWNLOAD"

# response = requests.get(url)
# response.raise_for_status()

# with open("../../scr/data/rat_sightings_data/Rat_Sightings_NYC.csv", "wb") as f:
#     f.write(response.content)

In [4]:
rs = pd.read_csv('../../scr/data/cleaned_rat_sightings_data/all_daily_borough_rs.csv')
rs['created_date'] = pd.to_datetime(rs['created_date']) 

# Start by cutting off data before 2020-01-01 and after 2026-02-28.
rs = rs[rs['created_date']<'2026-03-01']
rs = rs[rs['created_date']>='2020-01-01']

## Restrict to MANHATTAN

rs = rs[rs['borough']=='MANHATTAN']

## Drop the column with borough

rs = rs.drop(columns=['borough'])

## rename columns for prophet

rs.rename(columns={'created_date': 'ds', 'count': 'y'}, inplace=True)

The forecasted values are the naive forecast i.e. we take the last observed day and assume that these are good predictions for the next 14 days.
    
We really should find a way to get forecast data that's better though.

In [5]:
lat, lon = 40.7831, -73.9712
last_date = rs['ds'].max()
start = "2020-01-01"
end   = last_date.strftime("%Y-%m-%d")  # use last date of rs

url = (
    "https://archive-api.open-meteo.com/v1/archive"
    f"?latitude={lat}&longitude={lon}"
    f"&start_date={start}&end_date={end}"
    "&daily=temperature_2m_max,temperature_2m_min,temperature_2m_mean,"
    "apparent_temperature_max,apparent_temperature_min,apparent_temperature_mean,"
    "precipitation_sum,snowfall_sum"
    "&timezone=America/New_York"
)

response = requests.get(url)
data = response.json()

if 'error' in data:
    nd = pd.read_csv("weatherdata.csv")
    nd['ds'] = pd.to_datetime(nd['date'])
    wd = nd.drop(columns=['date'])
else:
    wd = pd.DataFrame(data["daily"])
    wd["ds"] = pd.to_datetime(wd["time"])
    wd = wd.drop(columns=["time"])

wd = wd.reset_index(drop=True)
future_dates = pd.date_range(start=last_date + pd.Timedelta(days=1),
                             periods=14,
                             freq='D')

last_row = wd.iloc[-1]
wd_14 = pd.DataFrame([last_row.values] * 14, columns=wd.columns)
wd_14['ds'] = future_dates 
wd_14 = wd_14.reset_index(drop=True)

In [6]:
# Suppress cmdstanpy info logs
logging.getLogger("cmdstanpy").setLevel(logging.WARNING)

regressed_features = ['apparent_temperature_max', 'apparent_temperature_min', 'snowfall_sum']

wd["ds"] = pd.to_datetime(wd["ds"])
wd_14["ds"] = pd.to_datetime(wd_14["ds"])

rs["ds"] = pd.to_datetime(rs["ds"])

rs = rs.merge(wd[['ds'] + regressed_features], on="ds", how="left")

### Fixing the Seed

We need to fix the seed for reproducibility. Without the following code block, the forecasts and *will* change on each run.

In [7]:
import random
import torch

# set seeds for reproducibility
seed = 42
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
torch.cuda.manual_seed_all(seed)

# make PyTorch deterministic
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

### Train the model

In [8]:
from neuralprophet import NeuralProphet
import pandas as pd

forecast_horizon = 14
regressed_features = ['apparent_temperature_max', 'apparent_temperature_min', 'snowfall_sum']

model = NeuralProphet(yearly_seasonality=True, 
                      weekly_seasonality=True, 
                      epochs=params['epochs'], 
                      learning_rate=params['learning_rate'], 
                      batch_size=params['batch_size'], 
                      n_forecasts = params['n_forecasts'], 
                      n_lags = params['n_lags'], 
                      ar_reg = params['ar_reg'], 
                      )
model = model.add_country_holidays(country_name="US")

for col in regressed_features:
    model.add_lagged_regressor(col, n_lags=params[col])

model.fit(rs, freq="D")
future = model.make_future_dataframe(rs, periods=forecast_horizon, n_historic_predictions=28)

for col in regressed_features:
    future[col] = pd.concat([rs[col], wd_14[col]], ignore_index=True)

forecast = model.predict(future)

rs_df = model.get_latest_forecast(forecast, include_previous_forecasts = 14)

rs_df

WARNING - (NP.forecaster.fit) - When Global modeling with local normalization, metrics are displayed in normalized scale.
INFO - (NP.df_utils._infer_frequency) - Major frequency D corresponds to 99.911% of the data.
INFO - (NP.df_utils._infer_frequency) - Defined frequency is equal to major frequency - D
INFO - (NP.data.processing._handle_missing_data) - Added 1 missing dates.
WARNING - (NP.data.processing._handle_missing_data) - 1 missing values in column y were detected in total. 
INFO - (NP.data.processing._handle_missing_data) - 1 NaN values in column y were auto-imputed.
WARNING - (NP.data.processing._handle_missing_data) - 1 missing values in column apparent_temperature_max were detected in total. 
INFO - (NP.data.processing._handle_missing_data) - 1 NaN values in column apparent_temperature_max were auto-imputed.
WARNING - (NP.data.processing._handle_missing_data) - 1 missing values in column apparent_temperature_min were detected in total. 
INFO - (NP.data.processing._handle_mi

Training: 0it [00:00, ?it/s]

INFO - (NP.df_utils._infer_frequency) - Major frequency D corresponds to 99.911% of the data.
INFO - (NP.df_utils._infer_frequency) - Defined frequency is equal to major frequency - D
WARNING - (py.warnings._showwarnmsg) - /opt/anaconda3/envs/erdos_ds_environment/lib/python3.12/site-packages/neuralprophet/data/split.py:273: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df = pd.concat([df, future_df])

INFO - (NP.df_utils.return_df_in_original_format) - Returning df with no ID column
INFO - (NP.df_utils._infer_frequency) - Major frequency D corresponds to 99.01% of the data.
INFO - (NP.df_utils._infer_frequency) - Defined frequency is equal to major frequency - D
INFO - (NP.df_utils._infer_frequency) - Major frequency D corresponds to 99.01%

Predicting: 10it [00:00, ?it/s]

INFO - (NP.df_utils.return_df_in_original_format) - Returning df with no ID column


,ds,y,origin-14,origin-13,origin-12,origin-11,origin-10,origin-9,origin-8,origin-7,origin-6,origin-5,origin-4,origin-3,origin-2,origin-1,origin-0
0,2026-02-15,9.0,5.166758,None,None,None,None,None,None,None,None,None,None,None,None,None,None
1,2026-02-16,7.0,8.884298,10.068938,None,None,None,None,None,None,None,None,None,None,None,None,None
2,2026-02-17,12.0,8.37533,10.400076,9.748915,None,None,None,None,None,None,None,None,None,None,None,None
3,2026-02-18,13.0,5.864586,8.469047,6.551909,7.186723,None,None,None,None,None,None,None,None,None,None,None
4,2026-02-19,10.0,7.197508,7.809831,7.745345,8.324533,9.362047,None,None,None,None,None,None,None,None,None,None
5,2026-02-20,12.0,8.598552,9.628661,9.476269,9.723001,9.996182,10.652277,None,None,None,None,None,None,None,None,None
6,2026-02-21,13.0,5.566334,5.497103,4.249698,5.407983,5.108604,6.540387,5.535063,None,None,None,None,None,None,None,None
7,2026-02-22,8.0,5.778276,7.497904,7.959683,7.34886,8.254622,8.448608,6.559163,9.326269,None,None,None,None,None,None,None
8,2026-02-23,3.0,12.716559,11.159421,10.982207,12.21303,13.291328,11.968062,10.911385,13.364361,12.890199,None,None,None,None,None,None
9,2026-02-24,7.0,11.447685,10.626505,12.749372,11.625265,11.204651,11.643913,12.08425,13.24637,13.040792,11.004862,None,None,None,None,None


### Forecast for 14 Days

In [9]:
df_wide= rs_df.copy()

# use the last 14 entries of 'origin-0' as the forecast
forecast_horizon = 14
origin_col = 'origin-0' 
last_14_preds = df_wide[origin_col].dropna().tail(forecast_horizon)

# compute corresponding forecast dates
last_14_dates = pd.to_datetime(df_wide['ds'].tail(forecast_horizon))

forecast_vertical = pd.DataFrame({'ds': last_14_dates, 'yhat': last_14_preds.values})

for _, row in forecast_vertical.iterrows():
    print(f"On day {row['ds'].date()}, the model predicts {round(row['yhat'])} rat sightings reported to 311.\n")

On day 2026-03-01, the model predicts 8 rat sightings reported to 311.

On day 2026-03-02, the model predicts 13 rat sightings reported to 311.

On day 2026-03-03, the model predicts 14 rat sightings reported to 311.

On day 2026-03-04, the model predicts 12 rat sightings reported to 311.

On day 2026-03-05, the model predicts 13 rat sightings reported to 311.

On day 2026-03-06, the model predicts 14 rat sightings reported to 311.

On day 2026-03-07, the model predicts 8 rat sightings reported to 311.

On day 2026-03-08, the model predicts 13 rat sightings reported to 311.

On day 2026-03-09, the model predicts 16 rat sightings reported to 311.

On day 2026-03-10, the model predicts 14 rat sightings reported to 311.

On day 2026-03-11, the model predicts 13 rat sightings reported to 311.

On day 2026-03-12, the model predicts 13 rat sightings reported to 311.

On day 2026-03-13, the model predicts 16 rat sightings reported to 311.

On day 2026-03-14, the model predicts 11 rat sighting

Example: On March 9th @ 11AM CST, I ran the above and got the following predictions.

On day 2026-03-01, the model predicts 8 rat sightings reported to 311.

On day 2026-03-02, the model predicts 13 rat sightings reported to 311.

On day 2026-03-03, the model predicts 14 rat sightings reported to 311.

On day 2026-03-04, the model predicts 12 rat sightings reported to 311.

On day 2026-03-05, the model predicts 13 rat sightings reported to 311.

On day 2026-03-06, the model predicts 14 rat sightings reported to 311.

On day 2026-03-07, the model predicts 8 rat sightings reported to 311.

On day 2026-03-08, the model predicts 13 rat sightings reported to 311.

On day 2026-03-09, the model predicts 16 rat sightings reported to 311.

On day 2026-03-10, the model predicts 14 rat sightings reported to 311.

On day 2026-03-11, the model predicts 13 rat sightings reported to 311.

On day 2026-03-12, the model predicts 13 rat sightings reported to 311.

On day 2026-03-13, the model predicts 16 rat sightings reported to 311.

On day 2026-03-14, the model predicts 11 rat sightings reported to 311.

In [10]:
model.plot(forecast)

ERROR - (NP.plotly.plot) - plotly-resampler is not installed. Please install it to use the resampler.
WARNING - (py.warnings._showwarnmsg) - /opt/anaconda3/envs/erdos_ds_environment/lib/python3.12/site-packages/neuralprophet/plot_forecast_plotly.py:100: FutureWarning: The behavior of DatetimeProperties.to_pydatetime is deprecated, in a future version this will return a Series containing python datetime objects instead of an ndarray. To retain the old behavior, call `np.array` on the result
  ds = fcst["ds"].dt.to_pydatetime()

WARNING - (py.warnings._showwarnmsg) - /opt/anaconda3/envs/erdos_ds_environment/lib/python3.12/site-packages/kaleido/_sync_server.py:11: UserWarning:




This means that static image generation (e.g. `fig.write_image()`) will not work.

Please upgrade Plotly to version 6.1.1 or greater, or downgrade Kaleido to version 0.2.1.





In [11]:
future = model.make_future_dataframe(rs)
forecast = model.predict(rs)
model.plot(forecast)


INFO - (NP.df_utils._infer_frequency) - Major frequency D corresponds to 99.911% of the data.
INFO - (NP.df_utils._infer_frequency) - Defined frequency is equal to major frequency - D
WARNING - (py.warnings._showwarnmsg) - /opt/anaconda3/envs/erdos_ds_environment/lib/python3.12/site-packages/neuralprophet/data/split.py:273: FutureWarning:

The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.


INFO - (NP.df_utils.return_df_in_original_format) - Returning df with no ID column
INFO - (NP.df_utils._infer_frequency) - Major frequency D corresponds to 99.911% of the data.
INFO - (NP.df_utils._infer_frequency) - Defined frequency is equal to major frequency - D
INFO - (NP.df_utils._infer_frequency) - Major frequency D corresponds to 99.911% of the data.
INFO - (NP.df_ut

Predicting: 10it [00:00, ?it/s]

INFO - (NP.df_utils.return_df_in_original_format) - Returning df with no ID column
ERROR - (NP.plotly.plot) - plotly-resampler is not installed. Please install it to use the resampler.
WARNING - (py.warnings._showwarnmsg) - /opt/anaconda3/envs/erdos_ds_environment/lib/python3.12/site-packages/neuralprophet/plot_forecast_plotly.py:100: FutureWarning:

The behavior of DatetimeProperties.to_pydatetime is deprecated, in a future version this will return a Series containing python datetime objects instead of an ndarray. To retain the old behavior, call `np.array` on the result


